[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BSLJunhyeonJeon/AI_COP/blob/main/session2/notebooks/01_data.ipynb)

# session2 · 01 · 데이터 준비 (혈액세포)

- **이 노트북에서 배우는 것**: 분류·감지·분할 세 태스크의 **데이터 형식 차이**를 눈으로 본다. (모델 학습/추론 없음 — 2단계 영역)
- **입력**: 없음 — 공개 데이터를 코드로 내려받습니다.
- **출력**: `data/{classification,detection,segmentation}` 채움 + `outputs/three_formats.png` (세 형식 비교 그림)

> **핵심 메시지**: 같은 혈액세포 이미지라도 **분류 → 감지 → 분할**로 갈수록 라벨이 정밀해지고 만드는 비용이 커진다.
> 데이터 출처(라이선스): BloodMNIST/MedMNIST(CC BY 4.0) · BCCD(MIT) · 분할 예시는 Otsu 임계처리(자체 생성).
> 위에서부터 셀을 **▶ 순서대로** 누르세요. (1단계는 GPU 불필요, 무료 CPU로 충분)

In [ ]:
# 셀 1 · 환경 감지 + 프로젝트 루트 확보 (00_setup 과 동일 패턴 — 분기는 이 셀 한 곳)
import os, subprocess

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

SESSION = "session2"
REPO_URL = "https://github.com/BSLJunhyeonJeon/AI_COP"
REPO_DIR = "/content/AI_COP"
SESSION_DIR = REPO_DIR + "/" + SESSION


def acquire_project():
    """코랩: 레포를 /content/AI_COP 로 클론 후 session2 를 PROJECT_ROOT 로 반환(실패 시 None)."""
    if os.path.isdir(REPO_DIR):
        print("이미 존재:", REPO_DIR, "(재클론 건너뜀)")
        try:
            r = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
            if r.returncode != 0:
                print("  (git pull 실패 — 기존 캐시 버전 사용)")
        except Exception as e:
            print("  (git pull 건너뜀:", e, ")")
    else:
        print("레포 클론:", REPO_URL, "->", REPO_DIR)
        try:
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
        except Exception as e:
            print("clone 실패(네트워크/권한 확인):", e)
    return SESSION_DIR if os.path.isdir(SESSION_DIR) else None


def find_root_local(marker="requirements.txt"):
    """로컬: requirements.txt 를 마커로 상위 탐색해 session2 루트를 찾는다."""
    start = os.path.abspath(os.getcwd())
    d = start
    while True:
        if os.path.exists(os.path.join(d, marker)):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            print("[주의] '" + marker + "' 를 못 찾음. 현재 폴더를 루트로 가정:", start)
            print("       (session2/ 안에서 노트북을 열었는지 확인하세요.)")
            return start
        d = parent


PROJECT_ROOT = acquire_project() if IN_COLAB else find_root_local()
if not (PROJECT_ROOT and os.path.isdir(PROJECT_ROOT)):
    raise RuntimeError(
        "세션 루트를 확보하지 못했습니다. "
        "코랩이면 레포 클론 실패이니 네트워크 확인 후 이 셀(셀 1)을 다시 ▶ 실행하세요. "
        "로컬이면 session2/ 안에서 노트북을 열었는지 확인하세요."
    )
os.chdir(PROJECT_ROOT)
print("실행 환경   :", "Colab" if IN_COLAB else "Local")
print("PROJECT_ROOT:", PROJECT_ROOT, "(이후 경로는 이 세션 루트 기준 상대경로)")

In [ ]:
# 셀 2 · 의존성 설치 (이번에 추가된 medmnist 등). 코랩 사전설치본은 재설치하지 않는다.
import os, sys, subprocess

if not os.path.exists("requirements.txt"):
    raise RuntimeError("requirements.txt 를 찾지 못했습니다. 셀 1을 먼저 ▶ 실행하세요.")
print("requirements.txt 설치 중... (medmnist 등 신규 의존성 포함)")
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=False)
if r.returncode != 0:
    raise RuntimeError("pip install 실패. 위 로그를 확인하고 네트워크/권한 점검 후 이 셀(셀 2)을 다시 ▶ 실행하세요.")

# 설치된 새 의존성 버전 출력 — 이 값을 requirements.txt 에 == 로 고정하세요.
print("\n[설치된 새 의존성 버전 — requirements.txt 핀에 사용]")
for mod in ["medmnist", "numpy", "matplotlib", "PIL"]:
    try:
        m = __import__(mod)
        name = "Pillow" if mod == "PIL" else mod
        print("  -", name, ":", getattr(m, "__version__", "?"))
    except Exception as e:
        print("  -", mod, ": import 실패 (", e, ")")

In [ ]:
# 셀 3 · 분류 데이터 (BloodMNIST / MedMNIST, CC BY 4.0)
# "폴더 이름 = 클래스" 구조로 소수 이미지를 PNG 저장한다. (공간 정보 없이 이미지 1장 = 라벨 1개)
import os, numpy as np
from PIL import Image
from medmnist import INFO, BloodMNIST

info = INFO["bloodmnist"]
label_map = info["label"]              # {'0':'basophil', ... '4':'lymphocyte','5':'monocyte','6':'neutrophil'}
PICK = [4, 5, 6]                       # 시각적으로 구분되는 3개: 림프구/단핵구/호중구
N_PER = 5                              # 클래스당 몇 장만 (가볍게)

# 64px 권장(가시성). 구버전 medmnist(size 인자 없음)는 28px 로 자동 폴백.
try:
    ds = BloodMNIST(split="train", download=True, size=64)
    print("BloodMNIST 로드 (size=64px)")
except TypeError:
    print("[주의] 이 medmnist 버전은 size 인자를 지원하지 않습니다 -> 28px 로 폴백합니다.")
    ds = BloodMNIST(split="train", download=True)
imgs, labels = ds.imgs, ds.labels.flatten()
print("  ->", len(imgs), "개 이미지, 한 장 shape =", imgs[0].shape)

base = os.path.join("data", "classification")
os.makedirs(base, exist_ok=True)
saved = {}
for cls in PICK:
    name = label_map[str(cls)].split("(")[0].strip().replace(" ", "_")
    cdir = os.path.join(base, name)
    os.makedirs(cdir, exist_ok=True)
    idxs = np.where(labels == cls)[0][:N_PER]
    for i, idx in enumerate(idxs):
        Image.fromarray(imgs[idx]).save(os.path.join(cdir, name + "_%02d.png" % i))
    saved[name] = int(len(idxs))

print("저장 위치: data/classification/<클래스명>/*.png")
print("=> 폴더 이름이 곧 라벨입니다 (이미지 1장 = 클래스 1개, 공간 정보 없음).")
if not saved:
    print("  [주의] 저장된 이미지가 0장입니다. 셀 2(설치)/네트워크를 확인 후 셀 3을 다시 ▶ 실행하세요.")
else:
    for name, n in saved.items():
        print("  -", name, ":", n, "장")
    print("  총", sum(saved.values()), "장")

In [ ]:
# 셀 4 · 감지 데이터 (BCCD, MIT) — 바운딩 박스 라벨(VOC XML)
# shallow clone 후 소수(2~3장)만 사용. 박스 형식: 클래스, xmin, ymin, xmax, ymax
import os, glob, shutil, subprocess
import xml.etree.ElementTree as ET

BCCD_DIR = os.path.join("data", "_bccd_src")   # data/ 아래 → .gitignore 로 비커밋
if not os.path.isdir(BCCD_DIR):
    print("BCCD shallow clone (전체 364장 중 2~3장만 사용)...")
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/Shenggan/BCCD_Dataset", BCCD_DIR], check=False)

src_img = os.path.join(BCCD_DIR, "BCCD", "JPEGImages")
src_ann = os.path.join(BCCD_DIR, "BCCD", "Annotations")
dst = os.path.join("data", "detection")
os.makedirs(dst, exist_ok=True)


def parse_voc(xml_path):
    root = ET.parse(xml_path).getroot()
    out = []
    for obj in root.findall("object"):
        b = obj.find("bndbox")
        out.append((obj.find("name").text,
                    int(b.find("xmin").text), int(b.find("ymin").text),
                    int(b.find("xmax").text), int(b.find("ymax").text)))
    return out


jpgs = sorted(glob.glob(os.path.join(src_img, "*.jpg")))[:3]
for jp in jpgs:
    stem = os.path.splitext(os.path.basename(jp))[0]
    shutil.copy(jp, os.path.join(dst, os.path.basename(jp)))
    xml = os.path.join(src_ann, stem + ".xml")
    if os.path.exists(xml):
        shutil.copy(xml, os.path.join(dst, stem + ".xml"))

print("저장 위치: data/detection/ (이미지 .jpg + 박스 라벨 .xml)")
if jpgs:
    stem = os.path.splitext(os.path.basename(jpgs[0]))[0]
    boxes = parse_voc(os.path.join(src_ann, stem + ".xml"))
    print("예시", stem, ": 박스", len(boxes), "개")
    print("형식: [클래스, xmin, ymin, xmax, ymax] = (x:xmin~xmax, y:ymin~ymax 사각형 안에 그 클래스 세포)")
    for bx in boxes[:8]:
        print("  ", bx)
    print("=> 물체마다 위치(사각형 좌표)+클래스. 분류보다 라벨이 정밀하고 비쌈.")
else:
    print("[주의] BCCD 이미지를 받지 못했습니다(클론 실패 가능).")
    print("       네트워크 확인 후 이 셀(셀 4)을 다시 ▶ 실행하세요. (미해결 시 셀 6 그림에 감지 칸이 빕니다)")

In [ ]:
# 셀 5 · 분할 형식 예시 — '마스크 = 원본과 같은 크기의 라벨 이미지(픽셀값=클래스)'
# 실제 분할 라벨링은 3단계에서 SAM2 가 혈액 이미지에 직접 수행. 여기선 '형식'만 보여준다.
# 가볍게: 셀3에서 받은 BloodMNIST 이미지 1장에 Otsu 임계처리(numpy 자체구현)로 예시 마스크 생성.
import os, glob, numpy as np
from PIL import Image


def otsu_threshold(gray):
    hist = np.histogram(gray, bins=256, range=(0, 256))[0].astype(float)
    total = gray.size
    sum_all = np.dot(np.arange(256), hist)
    sumB, wB, best_var, thr = 0.0, 0.0, -1.0, 0
    for t in range(256):
        wB += hist[t]
        if wB == 0:
            continue
        wF = total - wB
        if wF == 0:
            break
        sumB += t * hist[t]
        mB = sumB / wB
        mF = (sum_all - sumB) / wF
        var = wB * wF * (mB - mF) ** 2
        if var > best_var:
            best_var, thr = var, t
    return thr


seg = os.path.join("data", "segmentation")
os.makedirs(seg, exist_ok=True)
cands = glob.glob(os.path.join("data", "classification", "*", "*.png"))
if cands:
    img = Image.open(cands[0]).convert("RGB")
    gray = np.array(img.convert("L"))
    thr = otsu_threshold(gray)
    mask = (gray < thr).astype(np.uint8)          # 세포(어두움)=1, 배경=0
    img.save(os.path.join(seg, "example_image.png"))
    Image.fromarray(mask).save(os.path.join(seg, "example_mask.png"))               # 픽셀값 = 클래스(0/1)
    Image.fromarray((mask * 255).astype(np.uint8)).save(os.path.join(seg, "example_mask_view.png"))  # 사람 눈으로 보기용
    print("저장 위치: data/segmentation/ (example_image.png + example_mask.png)")
    print("마스크 shape:", mask.shape, "= 원본과 같은 크기")
    print("마스크 고유값:", np.unique(mask).tolist(), " (0=배경, 1=세포 — 픽셀값이 곧 클래스)")
    print("=> 라벨이 픽셀 단위. 가장 정밀하고 만들기 비쌈. (실제 라벨링은 3단계 SAM2)")
else:
    print("[주의] 분류 이미지가 없습니다. 셀 3을 먼저 실행하세요.")

In [ ]:
# 셀 6 · 세 형식 비교 시각화 (이 노트북의 핵심) — outputs/ 에 저장
# (그림 안 글자는 폰트 호환을 위해 영문. 한국어 설명은 아래 출력/마크다운에서.)
import os, glob, numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import xml.etree.ElementTree as ET

fig, ax = plt.subplots(1, 3, figsize=(13, 4.6))

# (1) 분류: 이미지 + 라벨 '글자만' (공간 정보 없음)
cls_imgs = sorted(glob.glob(os.path.join("data", "classification", "*", "*.png")))
if cls_imgs:
    label = os.path.basename(os.path.dirname(cls_imgs[0]))
    ax[0].imshow(Image.open(cls_imgs[0]))
    ax[0].set_title("Classification\n(label = a word)")
    ax[0].text(0.5, -0.10, 'label = "' + label + '"', transform=ax[0].transAxes,
               ha="center", color="crimson", fontsize=12)
ax[0].axis("off")

# (2) 감지: 이미지 + 바운딩 박스(+클래스)
det = sorted(glob.glob(os.path.join("data", "detection", "*.jpg")))
if det:
    ax[1].imshow(Image.open(det[0]))
    stem = os.path.splitext(os.path.basename(det[0]))[0]
    xmlp = os.path.join("data", "detection", stem + ".xml")
    if os.path.exists(xmlp):
        for obj in ET.parse(xmlp).getroot().findall("object"):
            b = obj.find("bndbox")
            x1, y1 = int(b.find("xmin").text), int(b.find("ymin").text)
            x2, y2 = int(b.find("xmax").text), int(b.find("ymax").text)
            ax[1].add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                            fill=False, edgecolor="lime", linewidth=1.3))
    ax[1].set_title("Detection\n(boxes = where + class)")
ax[1].axis("off")

# (3) 분할: 이미지 + 마스크 오버레이(픽셀 단위)
seg_img = os.path.join("data", "segmentation", "example_image.png")
seg_mask = os.path.join("data", "segmentation", "example_mask.png")
if os.path.exists(seg_img) and os.path.exists(seg_mask):
    base = np.array(Image.open(seg_img).convert("RGB"))
    m = np.array(Image.open(seg_mask))
    ov = base.copy()
    sel = m > 0
    ov[sel] = (0.5 * ov[sel] + 0.5 * np.array([255, 0, 0])).astype(np.uint8)
    ax[2].imshow(ov)
    ax[2].set_title("Segmentation\n(mask = per-pixel class)")
ax[2].axis("off")

fig.suptitle("Same blood cells - labels get more precise (and costlier):  class word  ->  box  ->  pixel mask",
             fontsize=12)
os.makedirs("outputs", exist_ok=True)
out = os.path.join("outputs", "three_formats.png")
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.savefig(out, dpi=130, bbox_inches="tight")
plt.show()
print("세 형식 비교 그림 저장:", out)
print("한 줄 요약: 같은 종류 이미지라도 왼쪽→오른쪽(분류→감지→분할)으로 라벨이 정밀해지고 비용이 커진다.")

In [ ]:
# 셀 7 · 검증 / 요약 — data/ 가 형식별로 채워졌는지 + 배운 것 3줄
import os, glob

classes = [d for d in glob.glob(os.path.join("data", "classification", "*")) if os.path.isdir(d)]
n_cls_imgs = len(glob.glob(os.path.join("data", "classification", "*", "*.png")))
n_det_img = len(glob.glob(os.path.join("data", "detection", "*.jpg")))
n_det_xml = len(glob.glob(os.path.join("data", "detection", "*.xml")))
n_seg = len(glob.glob(os.path.join("data", "segmentation", "*.png")))
fig_ok = os.path.exists(os.path.join("outputs", "three_formats.png"))

print("=" * 54)
print(" 1단계 · 데이터 준비 검증")
print("=" * 54)
print("- 분류 classification :", len(classes), "개 클래스 폴더,", n_cls_imgs, "장   (폴더 = 라벨)")
print("- 감지 detection      : 이미지", n_det_img, "장 + VOC XML", n_det_xml, "개   (박스 좌표)")
print("- 분할 segmentation   : 파일", n_seg, "개   (마스크 = 픽셀 라벨)")
print("- 비교 그림 outputs   :", "있음" if fig_ok else "없음", " (outputs/three_formats.png)")
print("=" * 54)
print("배운 것:")
print(" 1) 분류 = 이미지 1장에 라벨 1개(폴더로 구분)        — 라벨이 가장 쌈")
print(" 2) 감지 = 물체마다 박스 좌표(클래스 + xmin..ymax)   — 중간")
print(" 3) 분할 = 픽셀마다 클래스(원본 크기 마스크 이미지)  — 가장 정밀하고 비쌈")